# Lab 01 Solution: Coding Agent Anatomy

Understand the internal loop of an AI coding agent -- the five phases
that turn a natural-language request into working code.

**What you'll learn:**
- The 5 phases: Plan, Code, Test, Reflect, Iterate
- Tool registry data structures
- How the agent loop decides when to stop

No API key needed -- pure Python standard library.

In [ ]:
import os
import shutil
import json

WORKDIR = "/tmp/aidev-lab-02-01"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

## Step 1: The 5 Phases of a Coding Agent

| Phase | Purpose | Examples |
|---|---|---|
| 1. Plan | Analyse the request, break into sub-tasks | Read the codebase, identify files to change |
| 2. Code | Generate or modify source code | Write new functions, edit existing files |
| 3. Test | Run tests or linters to verify correctness | Execute unit tests, type-check, lint |
| 4. Reflect | Evaluate test results, diagnose failures | Parse error messages, compare expected vs actual |
| 5. Iterate | Loop back to Plan or Code if issues remain | Retry with improved approach, max N iterations |

The agent keeps looping through phases 1-5 until:
- a) All tests pass (success)
- b) Max iterations reached (give up gracefully)
- c) User interrupts (manual override)

In [ ]:
phases = [
    ("1. Plan",     "Analyse the request, break into sub-tasks",
     "Read the codebase, identify files to change"),
    ("2. Code",     "Generate or modify source code",
     "Write new functions, edit existing files"),
    ("3. Test",     "Run tests or linters to verify correctness",
     "Execute unit tests, type-check, lint"),
    ("4. Reflect",  "Evaluate test results, diagnose failures",
     "Parse error messages, compare expected vs actual"),
    ("5. Iterate",  "Loop back to Plan or Code if issues remain",
     "Retry with improved approach, max N iterations"),
]

## Step 2: Tool Registry

A coding agent needs tools to interact with the file system,
run commands, and search code. These are stored in a registry.

In [ ]:
example_registry = {
    "read_file":    {"description": "Read file contents",          "params": ["path"]},
    "write_file":   {"description": "Write content to file",       "params": ["path", "content"]},
    "run_tests":    {"description": "Execute test suite",          "params": ["test_path"]},
    "search_code":  {"description": "Grep for a pattern in files", "params": ["pattern", "directory"]},
    "list_files":   {"description": "List files in a directory",   "params": ["directory"]},
}

print(f"{'Tool':<14} {'Description':<35} {'Parameters'}")
print(f"{'-'*70}")
for name, info in example_registry.items():
    print(f"{name:<14} {info['description']:<35} {', '.join(info['params'])}")

## Step 3: Agent Loop Pseudocode

```python
def agent_loop(request, max_iterations=5):
    context = gather_context(request)       # read relevant files
    plan = create_plan(request, context)     # break into steps

    for i in range(max_iterations):
        code_changes = generate_code(plan)   # LLM writes code
        apply_changes(code_changes)          # write to disk

        test_results = run_tests()           # execute tests
        if test_results.all_passed:
            return "success"                 # done!

        diagnosis = reflect(test_results)    # analyse failures
        plan = revise_plan(plan, diagnosis)  # update the plan

    return "max iterations reached"          # give up
```

## TODO 1: Implement a `ToolRegistry` Class (5 points) -- Solution

Build a class that manages a collection of tools an agent can use.
This mirrors how real coding agents (like Claude Code, Cursor, etc.)
maintain a registry of available capabilities.

| Method | Signature | Behaviour |
|--------|-----------|-----------|
| `__init__` | `(self)` | Initialize with an empty `self._tools` dict |
| `register` | `(self, name: str, description: str, params: list)` | Add a tool: `{name: {"description": description, "params": params}}` |
| `list_tools` | `(self) -> list` | Return **sorted** list of tool names |
| `get_tool` | `(self, name: str) -> dict or None` | Return tool info dict, or `None` if not found |
| `count` | `(self) -> int` | Return number of registered tools |

In [ ]:
class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, name: str, description: str, params: list):
        self._tools[name] = {"description": description, "params": params}

    def list_tools(self) -> list:
        return sorted(self._tools.keys())

    def get_tool(self, name: str):
        return self._tools.get(name, None)

    def count(self) -> int:
        return len(self._tools)

In [ ]:
# Validation — TODO 1 (5 checks)
score1 = 0
checks_1 = []

try:
    reg = ToolRegistry()

    # Check 1: Registry starts empty
    if reg.count() == 0:
        checks_1.append(("Registry starts empty", "PASS")); score1 += 1
    else:
        checks_1.append(("Registry starts empty", "FAIL"))

    # Check 2: Can register and retrieve a tool
    reg.register("read_file", "Read file contents", ["path"])
    tool = reg.get_tool("read_file")
    if (isinstance(tool, dict) and tool.get("description") == "Read file contents"
            and tool.get("params") == ["path"]):
        checks_1.append(("Register and retrieve a tool", "PASS")); score1 += 1
    else:
        checks_1.append(("Register and retrieve a tool", "FAIL"))

    # Check 3: list_tools returns sorted names
    reg.register("write_file", "Write content to file", ["path", "content"])
    reg.register("search_code", "Grep for a pattern", ["pattern", "directory"])
    names = reg.list_tools()
    if names == ["read_file", "search_code", "write_file"]:
        checks_1.append(("list_tools returns sorted names", "PASS")); score1 += 1
    else:
        checks_1.append(("list_tools returns sorted names", "FAIL"))

    # Check 4: get_tool returns None for unknown tools
    if reg.get_tool("nonexistent_tool") is None:
        checks_1.append(("get_tool returns None for unknown", "PASS")); score1 += 1
    else:
        checks_1.append(("get_tool returns None for unknown", "FAIL"))

    # Check 5: count returns correct number
    if reg.count() == 3:
        checks_1.append(("count() returns 3 after 3 registrations", "PASS")); score1 += 1
    else:
        checks_1.append(("count() returns 3 after 3 registrations", "FAIL"))

except Exception as e:
    while len(checks_1) < 5:
        labels = ["Registry starts empty", "Register and retrieve a tool",
                   "list_tools returns sorted names", "get_tool returns None for unknown",
                   "count() returns 3 after 3 registrations"]
        checks_1.append((labels[len(checks_1)], "TODO"))

for check, status in checks_1:
    print(f"[{status}] {check}")

print(f"\nScore: {score1}/5")

## TODO 2: Implement an `AgentLoop` Simulator (4 points) -- Solution

Create a function that simulates the 5-phase coding agent loop
using mock functions for each phase.

```
simulate_agent_loop(task: str, max_iterations: int = 3) -> dict
```

**Behaviour per iteration `i` (0-based):**

| Phase | Mock output |
|-------|-------------|
| plan | `f"Plan for: {task} (attempt {i+1})"` |
| code | `f"Generated code for: {task}"` |
| test | `True` if `i == max_iterations - 1`, else `False` |
| reflect | `f"Diagnosis: adjusting approach for {task}"` (only when test is `False`) |

**Return:** `{"task", "iterations", "result", "history"}`

In [ ]:
def simulate_agent_loop(task: str, max_iterations: int = 3) -> dict:
    history = []
    result = "max_iterations_reached"

    for i in range(max_iterations):
        # Plan phase
        plan_output = f"Plan for: {task} (attempt {i+1})"
        history.append({"iteration": i + 1, "phase": "plan", "output": plan_output})

        # Code phase
        code_output = f"Generated code for: {task}"
        history.append({"iteration": i + 1, "phase": "code", "output": code_output})

        # Test phase — passes on the last iteration
        test_passed = (i == max_iterations - 1)
        history.append({"iteration": i + 1, "phase": "test", "output": str(test_passed)})

        if test_passed:
            result = "success"
            break

        # Reflect phase — only when test fails
        reflect_output = f"Diagnosis: adjusting approach for {task}"
        history.append({"iteration": i + 1, "phase": "reflect", "output": reflect_output})

    return {
        "task": task,
        "iterations": i + 1,
        "result": result,
        "history": history,
    }

In [ ]:
# Validation — TODO 2 (4 checks)
score2 = 0
checks_2 = []

try:
    result = simulate_agent_loop("fix login bug", max_iterations=3)

    # Check 1: Returns dict with required keys
    required_keys = {"task", "iterations", "result", "history"}
    if isinstance(result, dict) and required_keys.issubset(result.keys()):
        checks_2.append(("Returns dict with required keys", "PASS")); score2 += 1
    else:
        checks_2.append(("Returns dict with required keys", "FAIL"))

    # Check 2: History contains entries for all phases
    phase_names = {entry["phase"] for entry in result.get("history", [])}
    if {"plan", "code", "test"}.issubset(phase_names):
        checks_2.append(("History contains plan/code/test phases", "PASS")); score2 += 1
    else:
        checks_2.append(("History contains plan/code/test phases", "FAIL"))

    # Check 3: Result is "success" when tests eventually pass
    if result.get("result") == "success":
        checks_2.append(("Result is 'success' when tests pass", "PASS")); score2 += 1
    else:
        checks_2.append(("Result is 'success' when tests pass", "FAIL"))

    # Check 4: Respects max_iterations
    result2 = simulate_agent_loop("impossible task", max_iterations=1)
    if result2.get("iterations") == 1:
        checks_2.append(("Respects max_iterations limit", "PASS")); score2 += 1
    else:
        checks_2.append(("Respects max_iterations limit", "FAIL"))

except Exception as e:
    while len(checks_2) < 4:
        labels = ["Returns dict with required keys", "History contains plan/code/test phases",
                   "Result is 'success' when tests pass", "Respects max_iterations limit"]
        checks_2.append((labels[len(checks_2)], "TODO"))

for check, status in checks_2:
    print(f"[{status}] {check}")

print(f"\nScore: {score2}/4")

## TODO 3: Phase Transition Logic (3 points) -- Solution

Implement a function that determines the next phase in the agent loop
given the current phase and whether the test passed.

| Current Phase | Condition | Next Phase |
|---------------|-----------|------------|
| `"plan"` | -- | `"code"` |
| `"code"` | -- | `"test"` |
| `"test"` | `test_passed=True` | `"done"` |
| `"test"` | `test_passed=False` | `"reflect"` |
| `"reflect"` | -- | `"iterate"` |
| `"iterate"` | -- | `"plan"` |
| anything else | -- | `"plan"` (default) |

In [ ]:
def next_phase(current_phase: str, test_passed: bool = False) -> str:
    if current_phase == "plan":
        return "code"
    elif current_phase == "code":
        return "test"
    elif current_phase == "test":
        return "done" if test_passed else "reflect"
    elif current_phase == "reflect":
        return "iterate"
    elif current_phase == "iterate":
        return "plan"
    else:
        return "plan"

In [ ]:
# Validation — TODO 3 (3 checks)
score3 = 0
checks_3 = []

try:
    # Check 1: plan -> code
    if next_phase("plan") == "code":
        checks_3.append(("plan -> code", "PASS")); score3 += 1
    else:
        checks_3.append(("plan -> code", "FAIL"))

    # Check 2: test + failed -> reflect
    if next_phase("test", test_passed=False) == "reflect":
        checks_3.append(("test + failed -> reflect", "PASS")); score3 += 1
    else:
        checks_3.append(("test + failed -> reflect", "FAIL"))

    # Check 3: test + passed -> done
    if next_phase("test", test_passed=True) == "done":
        checks_3.append(("test + passed -> done", "PASS")); score3 += 1
    else:
        checks_3.append(("test + passed -> done", "FAIL"))

except Exception as e:
    while len(checks_3) < 3:
        labels = ["plan -> code", "test + failed -> reflect", "test + passed -> done"]
        checks_3.append((labels[len(checks_3)], "TODO"))

for check, status in checks_3:
    print(f"[{status}] {check}")

print(f"\nScore: {score3}/3")

## Save Reference Document

In [ ]:
ref = {
    "phases": ["plan", "code", "test", "reflect", "iterate"],
    "phase_details": {p[0].split(". ")[1]: p[1] for p in phases},
    "tool_registry_example": example_registry,
    "transitions": {
        "plan": "code", "code": "test",
        "test_passed": "done", "test_failed": "reflect",
        "reflect": "iterate", "iterate": "plan",
    },
}

with open(os.path.join(WORKDIR, "agent-anatomy-reference.json"), "w") as f:
    json.dump(ref, f, indent=2)

print(f"Reference saved to {WORKDIR}/agent-anatomy-reference.json")

## Summary

In [ ]:
total = score1 + score2 + score3
max_total = 5 + 4 + 3

print(f"TODO 1: {score1}/5 ToolRegistry checks passed")
print(f"TODO 2: {score2}/4 AgentLoop checks passed")
print(f"TODO 3: {score3}/3 phase transition checks passed")
print(f"\nTotal: {total}/{max_total}")
print(f"\nFiles generated in {WORKDIR}/")

### Key Takeaways

1. A **ToolRegistry** class encapsulates tool management (register, lookup, list)
2. The agent loop is an **iterative process**: plan -> code -> test -> reflect -> iterate
3. **Phase transitions** depend on test outcomes: pass leads to done, fail leads to reflect